In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from pprint import pprint

import src.utils as utils
import src.visuals as visual

from src.models import PINN
from src.loss import NavierStokesLoss
from src.dataloader import load_data, gen_dataloaders
from src.train import train_collocation

In [2]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

TRAIN_DATA_PATH = pathlib.Path("data/sampled_data/frac_1")
TRAIN_FILE_NAME = "data"

TEST_DATA_PATH = pathlib.Path("data/raw_data")
TEST_FILE_NAME = "data"

INPUT_COL_NAMES = ["time", "re", "x", "y"]
TARGET_COL_NAMES = ["U_x", "U_y", "p"]

BOX = {
    "x_min": 0.15,
    "x_max": 0.85,
    "y_min": 0.15,
    "y_max": 0.85,
}

EPOCHS = 300
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

N_COLLOCATION = 4096

PHYSICS_WEIGHTS = [
    0.0,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0
]

LR_FACTOR = 0.5
LR_PATIENCE = 15

print(f"Datasets: {TRAIN_FILE_NAME}, {TEST_FILE_NAME}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Collocation points: {N_COLLOCATION}")
print(f"Physics weights: {PHYSICS_WEIGHTS}")
print(f"Held-out box: {BOX}")

Device: cuda
Datasets: data, data
Epochs: 300
Batch size: 256
Collocation points: 4096
Physics weights: [0.0, 0.01, 0.05, 0.1, 0.5, 1.0]
Held-out box: {'x_min': 0.15, 'x_max': 0.85, 'y_min': 0.15, 'y_max': 0.85}


In [3]:
train_df, valid_df, _ = load_data(TRAIN_DATA_PATH, TRAIN_FILE_NAME)
_, _, test_df = load_data(TEST_DATA_PATH, TEST_FILE_NAME)

print(f"Train samples:      {len(train_df):,}")
print(f"Validation samples: {len(valid_df):,}")
print(f"Test samples:       {len(test_df):,}")

train_re = list(map(int, sorted(train_df["re"].unique())))
valid_re = list(map(int, sorted(valid_df["re"].unique())))
test_re = list(map(int, sorted(test_df["re"].unique())))

time_steps = list(map(float, sorted(train_df["time"].unique())))

print(f"\nTraining Reynolds numbers:   {train_re}")
print(f"Validation Reynolds numbers: {valid_re}")
print(f"Test Reynolds numbers:       {test_re}")

print(f"\nTime steps: {time_steps}")

Train samples:      2,364
Validation samples: 473
Test samples:       1,441,792

Training Reynolds numbers:   [100, 136, 155, 191, 210, 246, 265, 283, 338, 357, 375, 393, 412, 448, 485, 504, 540, 577, 595, 614, 632, 651, 687, 706, 724, 761, 779, 797, 816, 834, 908, 944, 963, 981, 1000]
Validation Reynolds numbers: [173, 302, 467, 522, 742, 871, 926]
Test Reynolds numbers:       [118, 228, 320, 430, 559, 669, 853, 889]

Time steps: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]


In [4]:
_, train_out = utils.split_by_box(train_df, BOX)

test_inside, test_outside = utils.split_by_box(test_df, BOX)

train_re_values = sorted(train_out["re"].unique())
valid_re_values = sorted(valid_df["re"].unique())

print(f"Original train samples: {len(train_df):,}")
print(f"Train after held-out:   {len(train_out):,}")
print(f"Removed from train:     {len(train_df) - len(train_out):,}")

print(f"\nValidation samples:     {len(valid_df):,}")

print(f"\nTest samples:")
print(f"  Inside box:           {len(test_inside):,}")
print(f"  Outside box:          {len(test_outside):,}")

removed_pct = 100 * (len(train_df) - len(train_out)) / len(train_df)

print(f"\nRemoved from training:  {removed_pct:.2f}%")

Original train samples: 2,364
Train after held-out:   1,250
Removed from train:     1,114

Validation samples:     473

Test samples:
  Inside box:           712,800
  Outside box:          728,992

Removed from training:  47.12%


In [5]:
mean = train_out.mean()
std = train_out.std()

train_df_norm = utils.normalize_data(train_out, mean, std)
valid_df_norm = utils.normalize_data(valid_df, mean, std)

In [6]:
train_dataloader, valid_dataloader, _ = gen_dataloaders(
    train_df_norm,
    valid_df_norm,
    valid_df_norm,
    INPUT_COL_NAMES,
    TARGET_COL_NAMES,
    BATCH_SIZE,
)

In [7]:
ranges = utils.get_domain_ranges(
    train_df,
    INPUT_COL_NAMES,
    overrides={
        "x": (0.0, 1.0),
        "y": (0.0, 1.0),
    },
)

In [ ]:
results = []

for c_physics in PHYSICS_WEIGHTS:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    print("\n" + "=" * 80)
    print(f"Training model with c_physics = {c_physics}")
    print("=" * 80)

    tag = f"cphys_{c_physics:g}"

    run_dir = utils.create_run_directory(label=f"heldout_{tag}")

    config = {
        "experiment": "heldout_region",
        "physics_weight": c_physics,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "collocation_points": N_COLLOCATION,
        "seed": SEED,
        "box": BOX,
        "train_samples": len(train_out),
        "valid_samples": len(valid_df),
        "test_inside_samples": len(test_inside),
        "test_outside_samples": len(test_outside),
    }

    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=4)

    model = PINN(
        len(INPUT_COL_NAMES),
        len(TARGET_COL_NAMES),
    ).to(device)

    criterion = NavierStokesLoss(
        c_physics,
        mean,
        std,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
    )

    history = train_collocation(
        model=model,
        train_dataloader=train_dataloader,
        valid_dataloader=valid_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=EPOCHS,
        run_dir=run_dir,
        input_col_names=INPUT_COL_NAMES,
        ranges=ranges,
        mean=mean,
        std=std,
        n_collocation=N_COLLOCATION,
        train_re_values=train_re_values,
        valid_re_values=valid_re_values,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)

    fig, _ = utils.plot_training_history(
        history_df,
        output_path=run_dir / "losses.png",
        title=f"Held-out (c_physics={c_physics})",
    )
    plt.close(fig)

    checkpoint = torch.load(
        run_dir / "best_model.pth",
        map_location=device,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    metrics = []

    for region_name, region_df in [
        ("inside_box", test_inside),
        ("outside_box", test_outside),
    ]:

        region_metrics = utils.evaluate_model(
            model=model,
            df=region_df,
            input_col_names=INPUT_COL_NAMES,
            target_col_names=TARGET_COL_NAMES,
            mean=mean,
            std=std,
            device=device,
        )

        row = {
            "tag": tag,
            "c_physics": c_physics,
            "region": region_name,
            "n_points": len(region_df),
        }

        row.update(utils.flatten_metrics(region_metrics))

        metrics.append(row)
        results.append(row)

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(run_dir / "metrics.csv", index=False)

    print(metrics_df[["region", "R2_all", "MAE_all", "RMSE_all"]])

In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["c_physics", "region"]
).reset_index(drop=True)

results_df.to_csv("heldout_results.csv", index=False)

display(results_df)

,tag,c_physics,region,n_points,MAE_all,MSE_all,RMSE_all,R2_all,MaxAbsError_all,RelL2_all,...,RMSE_U_y,R2_U_y,MaxAbsError_U_y,RelL2_U_y,MAE_p,MSE_p,RMSE_p,R2_p,MaxAbsError_p,RelL2_p
0,cphys_0,0.00,inside_box,712800,0.020861,0.001239,0.035196,0.864928,0.210064,0.359359,...,0.026699,0.945984,0.143911,0.228576,0.007150,0.000162,0.012710,0.803546,0.059296,0.355125
1,cphys_0,0.00,outside_box,728992,0.008885,0.000735,0.027114,0.966756,2.827594,0.181781,...,0.021791,0.974309,0.404957,0.158465,0.003549,0.000654,0.025564,0.787848,2.827594,0.458969
2,cphys_0.01,0.01,inside_box,712800,0.016190,0.000704,0.026535,0.923224,0.173141,0.270931,...,0.025528,0.950615,0.137259,0.218558,0.007030,0.000166,0.012866,0.798684,0.055831,0.359492
3,cphys_0.01,0.01,outside_box,728992,0.007915,0.000672,0.025926,0.969608,2.839279,0.173810,...,0.020605,0.977028,0.382888,0.149846,0.003266,0.000661,0.025710,0.785423,2.839279,0.461585
4,cphys_0.05,0.05,inside_box,712800,0.014012,0.000504,0.022448,0.945056,0.158465,0.229195,...,0.025585,0.950397,0.117113,0.219040,0.005664,0.000103,0.010129,0.875231,0.048721,0.283011
5,cphys_0.05,0.05,outside_box,728992,0.007768,0.000631,0.025119,0.971468,2.791082,0.168406,...,0.021082,0.975953,0.343704,0.153312,0.003281,0.000650,0.025501,0.788897,2.791082,0.457833
6,cphys_0.1,0.10,inside_box,712800,0.013245,0.000476,0.021812,0.948126,0.144542,0.222700,...,0.024463,0.954651,0.106661,0.209437,0.005265,0.000091,0.009562,0.888808,0.047138,0.267170
7,cphys_0.1,0.10,outside_box,728992,0.007648,0.000637,0.025233,0.971209,2.792112,0.169169,...,0.021219,0.975639,0.338719,0.154309,0.003209,0.000666,0.025806,0.783818,2.792112,0.463308
8,cphys_0.5,0.50,inside_box,712800,0.011812,0.000379,0.019464,0.958692,0.123451,0.198729,...,0.021553,0.964799,0.114264,0.184523,0.004752,0.000069,0.008280,0.916624,0.041838,0.231351
9,cphys_0.5,0.50,outside_box,728992,0.008911,0.000696,0.026383,0.968525,2.771634,0.176880,...,0.022618,0.972322,0.398576,0.164479,0.003619,0.000684,0.026161,0.777820,2.771634,0.469691
